In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

## Daten

In [ ]:
folds = pd.read_csv("../out/gridsearch/folds.csv")
results = pd.read_csv("../out/gridsearch/results.csv")

# Configs von der besten zur schlechtesten, fuer die Reihenfolge in allen Plots
order = results.sort_values("mean_val_accuracy", ascending=False)["config_id"].tolist()

print(folds.shape, results.shape)
folds.head()

In [ ]:
cols = [
    "config_id", "dims", "optimizer", "eta_infer", "T_infer", "lr", "weight_decay",
    "negative_slope", "batch_size", "n_params",
    "mean_val_accuracy", "std_val_accuracy",
    "mean_val_f1_macro", "std_val_f1_macro",
    "mean_val_cross_entropy", "std_val_cross_entropy",
    "mean_total_time_s", "std_total_time_s",
]
results[cols].round(4)

## Verteilung Folds

In [ ]:
plt.figure(figsize=(14, 5))
sns.boxplot(data=folds, x="config_id", y="val_accuracy", order=order, color="lightsteelblue")
sns.stripplot(data=folds, x="config_id", y="val_accuracy", order=order, color="black", size=5)

plt.title("Validation Accuracy pro Config (5 Folds)")
plt.xlabel("config_id (sortiert nach mean_val_accuracy)")
plt.ylabel("Validation Accuracy")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
metrics = ["val_accuracy", "val_f1_macro", "val_cross_entropy", "total_time_s"]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, metric in zip(axes.ravel(), metrics):
    sns.boxplot(data=folds, x="config_id", y=metric, order=order, ax=ax, color="lightsteelblue")
    sns.stripplot(data=folds, x="config_id", y=metric, order=order, ax=ax, color="black", size=3)
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## Hyperparameter vs Accuracy

In [ ]:
params = [
    "eta_infer", "T_infer", "lr", "weight_decay",
    "negative_slope", "batch_size", "hidden_width", "n_hidden_layers",
]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, param in zip(axes.ravel(), params):
    ax.errorbar(
        results[param], results["mean_val_accuracy"], yerr=results["std_val_accuracy"],
        fmt="o", capsize=3, alpha=0.8,
    )
    ax.set_xlabel(param)
    ax.set_ylabel("mean_val_accuracy")
    if param in ("lr", "weight_decay", "negative_slope"):
        ax.set_xscale("log")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Accuracy gegen Laufzeit: was kostet die beste Config?
plt.figure(figsize=(8, 6))
for optimizer, group in results.groupby("optimizer"):
    plt.errorbar(
        group["mean_total_time_s"], group["mean_val_accuracy"],
        yerr=group["std_val_accuracy"], xerr=group["std_total_time_s"],
        fmt="o", capsize=3, label=optimizer, alpha=0.8,
    )

for row in results.itertuples():
    plt.annotate(row.config_id, (row.mean_total_time_s, row.mean_val_accuracy),
                 textcoords="offset points", xytext=(6, 3), fontsize=8)

plt.xlabel("mean_total_time_s (pro Fold)")
plt.ylabel("mean_val_accuracy")
plt.title("Accuracy gegen Laufzeit")
plt.legend(title="optimizer")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Hyperparameter vs F1